In [ ]:
from pathlib import Path

from plistsync.core.rewrite import PathRewrite
from plistsync.services import traktor

Edit the config below

In [ ]:
collection_path = "/Users/paul/Music/Traktor/collection.nml"

# Im using traktor on windows on an external drive D:/SYNC
# and running the script on my linux machine
rewrite = PathRewrite.from_str(old="/D:/SYNC", new="/mnt/media/music")

outdir = "./out"

In [ ]:
collection = traktor.NMLCollection(collection_path)

List all playlist in collection. At the moment we do not have a way to show the collection structure.

In [ ]:
for pl in collection.playlists():
    print(pl.name)

In [ ]:
# Select a playlist
plist_name = "synthetic roots"
pl = collection.playlist(plist_name)

I want to create a directory for the output converted music.

In [ ]:
out_pl_dir = Path(outdir) / plist_name
out_pl_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Create local tracks from the tracks to write to new locations

from plistsync.services.local import LocalTrack
from plistsync.services.traktor import NMLPlaylistTrack


def to_local_track(track: NMLPlaylistTrack) -> LocalTrack:
    """Convert a NMLTrack to a local track."""
    local_path = rewrite.apply(track.path)
    return LocalTrack(
        path=local_path,
    )


local_tracks = [to_local_track(t) for t in pl]

Convert all files from flac to mp3 as cdjs normally cant play flac
`$ pip install python-ffmpeg`

In [ ]:
from ffmpeg import FFmpeg


# Convert to mp3 if not already mp3 using ffmpeg
def _transcode_to_mp3(input_path: Path, output_path: Path) -> None:
    if output_path.exists():
        return

    pipe = (
        FFmpeg()
        .option("ab", "320k")  # Set audio bitrate
        .option("map_metadata", "0")
        .option("id3v2_version", "3")
        .input(str(input_path))
        .output(str(output_path))
    )
    pipe.execute()


def to_mp3_local_track(track: LocalTrack, out_dir: Path) -> LocalTrack:
    """Convert a local track to a mp3 local track."""
    out_path = out_dir / track.path.name.replace(".flac", ".mp3")
    _transcode_to_mp3(track.path, out_path)
    return LocalTrack(
        path=out_path,
    )

In [ ]:
mp3_tracks = [to_mp3_local_track(t, out_pl_dir) for t in local_tracks]

In [ ]:
# Create simple m3u with all files
m3u_path = out_pl_dir / f"{plist_name}.m3u"
with m3u_path.open("w") as f:
    for track in mp3_tracks:
        f.write(f"./{track.path.relative_to(out_pl_dir)}\n")